## 🎯 Kernel Tutorial: Adding 10 with Boundary Conditions 🚀

### 🎬 Video Overview
Learn how to implement a CUDA kernel that adds 10 to each element of a vector/matrix, but with a twist - handling cases where you have fewer data elements than threads! 🔥

### 🧠 The Core Concept
📝 What We're Building
A kernel that:

- ✅ Takes input vector/matrix a
- ✅ Adds 10 to each element
- ✅ Stores result in output
- ⚠️ But: Handles boundary conditions when threads > data elements

## Until now

<img src="../../assets/004_thread_mgmt-1.png" width="600" height="600">

## Today

<img src="../../assets/004_thread_mgmt-2.png" width="800" height="600">

In [1]:
import mojo.notebook

In [6]:
%%mojo

from memory import UnsafePointer
from gpu import thread_idx
from gpu.host import DeviceContext


alias SIZE = 4
alias BLOCKS_PER_GRID = 1
alias THREADS_PER_BLOCK = (8, 1)
alias dtype = DType.float32


fn add_10(
    output: UnsafePointer[Scalar[dtype]],
    a: UnsafePointer[Scalar[dtype]],
    size: Int,
):
    i = thread_idx.x

    # 🛡️ CRITICAL BOUNDARY CHECK!
    if i < size:  # 🚫 Threads with idx >= n do nothing and return safely
        output[i] = a[i] + 10.0


def main():

    # BOILER PLATE
    var ctx = DeviceContext()
    out = ctx.enqueue_create_buffer[dtype](SIZE)
    out.enqueue_fill(0)
    a = ctx.enqueue_create_buffer[dtype](SIZE)
    a.enqueue_fill(0)
    with a.map_to_host() as a_host:
        for i in range(SIZE):
            a_host[i] = i

    ## call the kernel - Use enqueue_function instead of enqueue_function_checked
    ctx.enqueue_function[add_10](
            out,
            a,
            SIZE,
            grid_dim=BLOCKS_PER_GRID,
            block_dim=THREADS_PER_BLOCK,
        )
    
    print(out)

MojoCompilationError: Error compiling Mojo at /tmp/tmpb80ag892/cell.mojo. Command: run /tmp/tmpb80ag892/cell.mojo

/tmp/tmpb80ag892/cell.mojo:21:12: warning: deprecated implicit conversion from 'Int' to 'UInt'
    if i < size:  # 🚫 Threads with idx >= n do nothing and return safely
/tmp/tmpb80ag892/cell.mojo:21:12: note: call 'UInt(...)' explicitly
    if i < size:  # 🚫 Threads with idx >= n do nothing and return safely
/tmp/tmpb80ag892/cell.mojo:1:1: note: '@implicit' constructor 'UInt.__init__' declared here

^
/tmp/tmpb80ag892/cell.mojo:22:15: error: expression must be mutable in assignment
        output[i] = a[i] + 10.0
        ~~~~~~^~~
/tmp/tmpb80ag892/cell.mojo:38:33: warning: `enqueue_function` is deprecated. Use `enqueue_function_checked` instead.
    ctx.enqueue_function[add_10](
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
/tmp/tmpb80ag892/cell.mojo:1:1: note: 'enqueue_function' declared here

^
/home/ablearn/mojo-gpu-tutorials/.pixi/envs/default/bin/mojo: error: failed to parse the p

### 2nd Kernel

<img src="../../assets/004_thread_mgmt-3.png" width="600" height="400">


### 📐 Visual Layout of BLOCKS, THREADS within a GRID

```
Thread  |    0  1  2  3
-------------------------
block 0 | [  0  1  2  3 ]   ← IDs: 0, 1, 2, 3
block 1 | [  0  1  2  3 ]   ← IDs: 4, 5, 6, 7
block 2 | [  0  1  2  3 ]   ← IDs: 8, 9, 10, 11  
block 3 | [  0  1  2  3 ]   ← IDs: 12, 13, 14, 15
```

In [11]:
%%mojo

from gpu import thread_idx, block_idx, block_dim
from memory import UnsafePointer
from gpu import thread_idx
from gpu.host import DeviceContext


alias SIZE_2k = 9
alias BLOCKS_PER_GRID_2k = (3, 1)
alias THREADS_PER_BLOCK_2k = (4, 1)
alias dtype_2k = DType.float32


fn add_10_2k(
    output: UnsafePointer[Scalar[dtype]],
    a: UnsafePointer[Scalar[dtype]],
    size: Int,
):
    i = block_dim.x * block_idx.x + thread_idx.x #<<< this is the only change. GEt the index correctly considering multiple blocks
    if i < size:   
        output[i] = a[i] + 10.0  

print("Adding Kernel Here")



def main():

    #BOILER PLATE 
    var ctx_2k = DeviceContext()
    out_2k = ctx_2k.enqueue_create_buffer[dtype_2k](SIZE_2k)
    out_2k.enqueue_fill(0)
    a_2k = ctx_2k.enqueue_create_buffer[dtype_2k](SIZE_2k)
    a_2k.enqueue_fill(0)
    with a_2k.map_to_host() as a_2k_host:
        for i in range(SIZE_2k):
            a_2k_host[i] = i

    print(a_2k)

    #BOILER PLATE 
    #CALL KERNEL 2
    ctx_2k.enqueue_function[add_10_2k](
                out_2k,
                a_2k,
                SIZE_2k,
                grid_dim=BLOCKS_PER_GRID_2k,
                block_dim=THREADS_PER_BLOCK_2k,
            )
    print(out_2k)

MojoCompilationError: Error compiling Mojo at /tmp/tmpsqiymqn3/cell.mojo. Command: run /tmp/tmpsqiymqn3/cell.mojo

/tmp/tmpsqiymqn3/cell.mojo:23:1: error: expressions are not supported at the file scope
print("Adding Kernel Here")
^
/tmp/tmpsqiymqn3/cell.mojo:15:34: error: use of unknown declaration 'dtype'
    output: UnsafePointer[Scalar[dtype]],
                                 ^~~~~
/tmp/tmpsqiymqn3/cell.mojo:16:29: error: use of unknown declaration 'dtype'
    a: UnsafePointer[Scalar[dtype]],
                            ^~~~~
/home/ablearn/mojo-gpu-tutorials/.pixi/envs/default/bin/mojo: error: failed to parse the provided Mojo source module


## 3rd Kernel

Apply the 2 priniciples

- convert the local ID -> global ID
- Use the "i<size" but in 2 dimensions

<img src="../../assets/004_thread_mgmt-4.png" width="600" height="400">

```

┌─────────────────────────────────────┬─────────────────────────────────────┐
│         Block (0,0)                 │         Block (1,0)                 │
│  ┌──────────┬──────────┬──────────┐ │  ┌──────────┬──────────┬──────────┐ │
│  │ B(0,0)   │ B(0,0)   │ B(0,0)   │ │  │ B(1,0)   │ B(1,0)   │ B(1,0)   │ │
│  │ T(0,0)   │ T(1,0)   │ T(2,0)   │ │  │ T(0,0)   │ T(1,0)   │ T(2,0)   │ │
│  │ ───────  │ ───────  │ ───────  │ │  │ ───────  │ ───────  │ ───────  │ │
│  │ a[0,0]   │ a[0,1]   │ a[0,2]   │ │  │ a[0,3]   │ a[0,4]   │ OUT❌    │ │
│  ├──────────┼──────────┼──────────┤ │  ├──────────┼──────────┼──────────┤ │
│  │ B(0,0)   │ B(0,0)   │ B(0,0)   │ │  │ B(1,0)   │ B(1,0)   │ B(1,0)   │ │
│  │ T(0,1)   │ T(1,1)   │ T(2,1)   │ │  │ T(0,1)   │ T(1,1)   │ T(2,1)   │ │
│  │ ───────  │ ───────  │ ───────  │ │  │ ───────  │ ───────  │ ───────  │ │
│  │ a[1,0]   │ a[1,1]   │ a[1,2]   │ │  │ a[1,3]   │ a[1,4]   │ OUT❌    │ │
│  ├──────────┼──────────┼──────────┤ │  ├──────────┼──────────┼──────────┤ │
│  │ B(0,0)   │ B(0,0)   │ B(0,0)   │ │  │ B(1,0)   │ B(1,0)   │ B(1,0)   │ │
│  │ T(0,2)   │ T(1,2)   │ T(2,2)   │ │  │ T(0,2)   │ T(1,2)   │ T(2,2)   │ │
│  │ ───────  │ ───────  │ ───────  │ │  │ ───────  │ ───────  │ ───────  │ │
│  │ a[2,0]   │ a[2,1]   │ a[2,2]   │ │  │ a[2,3]   │ a[2,4]   │ OUT❌    │ │
│  └──────────┴──────────┴──────────┘ │  └──────────┴──────────┴──────────┘ │
├─────────────────────────────────────┼─────────────────────────────────────┤
│         Block (0,1)                 │         Block (1,1)                 │
│  ┌──────────┬──────────┬──────────┐ │  ┌──────────┬──────────┬──────────┐ │
│  │ B(0,1)   │ B(0,1)   │ B(0,1)   │ │  │ B(1,1)   │ B(1,1)   │ B(1,1)   │ │
│  │ T(0,0)   │ T(1,0)   │ T(2,0)   │ │  │ T(0,0)   │ T(1,0)   │ T(2,0)   │ │
│  │ ───────  │ ───────  │ ───────  │ │  │ ───────  │ ───────  │ ───────  │ │
│  │ a[3,0]   │ a[3,1]   │ a[3,2]   │ │  │ a[3,3]   │ a[3,4]   │ OUT❌    │ │
│  ├──────────┼──────────┼──────────┤ │  ├──────────┼──────────┼──────────┤ │
│  │ B(0,1)   │ B(0,1)   │ B(0,1)   │ │  │ B(1,1)   │ B(1,1)   │ B(1,1)   │ │
│  │ T(0,1)   │ T(1,1)   │ T(2,1)   │ │  │ T(0,1)   │ T(1,1)   │ T(2,1)   │ │
│  │ ───────  │ ───────  │ ───────  │ │  │ ───────  │ ───────  │ ───────  │ │
│  │ a[4,0]   │ a[4,1]   │ a[4,2]   │ │  │ a[4,3]   │ a[4,4]   │ OUT❌    │ │
│  ├──────────┼──────────┼──────────┤ │  ├──────────┼──────────┼──────────┤ │
│  │ B(0,1)   │ B(0,1)   │ B(0,1)   │ │  │ B(1,1)   │ B(1,1)   │ B(1,1)   │ │
│  │ T(0,2)   │ T(1,2)   │ T(2,2)   │ │  │ T(0,2)   │ T(1,2)   │ T(2,2)   │ │
│  │ ───────  │ ───────  │ ───────  │ │  │ ───────  │ ───────  │ ───────  │ │
│  │ OUT❌    │ OUT❌    │ OUT❌    │ │  │ OUT❌    │ OUT❌    │ OUT❌    │ │
│  └──────────┴──────────┴──────────┘ │  └──────────┴──────────┴──────────┘ │
└─────────────────────────────────────┴─────────────────────────────────────┘

Legend:
  B(x,y) = Block Index (block_idx.x, block_idx.y)
  T(x,y) = Thread Index (thread_idx.x, thread_idx.y)
  a[r,c] = Matrix Element at [row, col]
  OUT❌  = Out of bounds for 5×5 matrix


```

In [12]:

%%mojo

from gpu import thread_idx, block_idx, block_dim
from memory import UnsafePointer
from gpu import thread_idx
from gpu.host import DeviceContext


alias SIZE_3k = 5
alias BLOCKS_PER_GRID_3k = (2, 2)
alias THREADS_PER_BLOCK_3k = (3, 3)
alias dtype_3k = DType.float32


fn add_10_3k(
    output: UnsafePointer[Scalar[dtype]],
    a: UnsafePointer[Scalar[dtype]],
    size: Int,
):
    row = block_dim.y * block_idx.y + thread_idx.y
    col = block_dim.x * block_idx.x + thread_idx.x
    
    if row < size and col < size:
        output[row * size + col] = a[row * size + col] + 10.0

# output[0 * 5 + 0] = output[0] = a[0] + 10
# output[0 * 5 + 1] = output[1] = a[1] + 10
# output[0 * 5 + 2] = output[2] = a[2] + 10
# output[0 * 5 + 3] = output[3] = a[3] + 10
# output[0 * 5 + 4] = output[4] = a[4] + 10

# second row
## output[1 * 5 + 0] = output[5] = a[5] + 10
## output[1 * 5 + 1] = output[6] = a[6] + 10
...

print("Adding Kernel Here")


#BOILER PLATE 
var ctx_3k = DeviceContext()
out_3k = ctx_3k.enqueue_create_buffer[dtype_3k](SIZE_3k * SIZE_3k)
out_3k.enqueue_fill(0)
a_3k = ctx_3k.enqueue_create_buffer[dtype_3k](SIZE_3k * SIZE_3k)
a_3k.enqueue_fill(0)
with a_3k.map_to_host() as a_3k_host:
    for j in range(SIZE_3k):
        for i in range(SIZE_3k):
            k = j * SIZE_3k + i
            a_3k_host[k] = k

print(a_3k)

ctx_3k.enqueue_function[add_10_3k](
            out_3k,
            a_3k,
            SIZE_3k,
            grid_dim=BLOCKS_PER_GRID_3k,
            block_dim=THREADS_PER_BLOCK_3k,
        )

ctx_3k.synchronize()

with out_3k.map_to_host() as out3k_host:
    for i in range(SIZE_3k):
        for j in range(SIZE_3k):
            print(out3k_host[i * SIZE_3k + j])


MojoCompilationError: Error compiling Mojo at /tmp/tmpzkbdm0jx/cell.mojo. Command: run /tmp/tmpzkbdm0jx/cell.mojo

/tmp/tmpzkbdm0jx/cell.mojo:36:1: error: expressions are not supported at the file scope
print("Adding Kernel Here")
^
/tmp/tmpzkbdm0jx/cell.mojo:40:1: error: global vars are not supported
var ctx_3k = DeviceContext()
^
/tmp/tmpzkbdm0jx/cell.mojo:41:1: error: expressions are not supported at the file scope
out_3k = ctx_3k.enqueue_create_buffer[dtype_3k](SIZE_3k * SIZE_3k)
^
/tmp/tmpzkbdm0jx/cell.mojo:42:1: error: expressions are not supported at the file scope
out_3k.enqueue_fill(0)
^
/tmp/tmpzkbdm0jx/cell.mojo:43:1: error: expressions are not supported at the file scope
a_3k = ctx_3k.enqueue_create_buffer[dtype_3k](SIZE_3k * SIZE_3k)
^
/tmp/tmpzkbdm0jx/cell.mojo:44:1: error: expressions are not supported at the file scope
a_3k.enqueue_fill(0)
^
/tmp/tmpzkbdm0jx/cell.mojo:45:6: error: use of unknown declaration 'a_3k'
with a_3k.map_to_host() as a_3k_host:
     ^~~~
/tmp/tmp

## 🎯 Key Takeaways: Handling Thread-Data Mismatches

- **Single block**
    - Get Thread Index, Then Thread Index < Size of vector
- **Multiple Blocks**
    - Get global Thread index, Then Thread Index < Size of vector
- **Mulitple Blocks Multi-Dimension**
    - Get Global Row Index
    - Get Global Column Index
    - Ensure both are less than row and column len of the matrix